# 7.15 學會新任務會忘記舊任務嗎？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩種遮罩：看得見，不一定直接考它**

X 的 assistant 格預測 A；PAD 不當答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/masks.svg")))

**先想一想：**只評新任務B，會遺漏哪種退步？

共享權重學新任務時，舊任務使用的參數也會改變。先建立A能力，再微調B，前後各測A和B；否則看不到遺忘。

**把直覺寫成數學：**記錄(A_before,B_before,A_after,B_after)，不能只報B_after。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
measurements = {"before": {"A": None, "B": None}, "after_B_only": {"A": None, "B": None}}
print(measurements)
print("執行配方：先train --task text保存A；再--task sft --checkpoint A.pt保存B；兩個checkpoint都評A/B")

**如何讀結果：**None誠實表示尚未訓練；學習遺忘機制不需要僞造實驗曲線。

**只改一件事：**只新增A的held-out問題列表，不更改訓練B。
